<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/protothema_technology_2026_scraper_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q requests beautifulsoup4 pandas numpy matplotlib spacy
!python -m spacy download el_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 25.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('el_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# scraping
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
import xml.etree.ElementTree as ET   # για να διαβάσουμε το RSS feed (είναι απλό XML)

# για τα nan values
import numpy as np

# για τα γραφήματα
import matplotlib.pyplot as plt

# για την επεξεργασία ελληνικών κειμένων (χρήσιμο αργότερα για ανάλυση, όχι για το scraping)
import spacy


In [ ]:
main_url = "https://www.protothema.gr"
category_url = "https://www.protothema.gr/technology/"
rss_url = "https://www.protothema.gr/technology/rss/"

target_year = 2026

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

# Boilerplate phrases -- μόλις εμφανιστεί μία από αυτές μέσα σε παράγραφο, σταματάμε να
# μαζεύουμε κείμενο (ίδια λογική με το STOP_PHRASES του tovima παραδείγματος).
STOP_PHRASES = [
    "Σχετικά Άρθρα",
    "ΡΟΗ ΕΙΔΗΣΕΩΝ",
    "ΤΑ ΠΙΟ ΔΗΜΟΦΙΛΗ",
    "Δείτε Επίσης",
    "Best of Network",
    "Thema Insights",
]


## ΒΗΜΑ Α: Μαζεύουμε τα urls των άρθρων (teasers) μέσω RSS

Ρόλος ίδιος με τη λούπα σελίδων του efsyn tutorial: θέλουμε μια λίστα από urls άρθρων. Το RSS
feed μας τη δίνει με ένα request αντί για πολλές σελίδες.

In [ ]:
response = requests.get(rss_url, headers=HEADERS)
root = ET.fromstring(response.content)

teasers_list = []
for item in root.findall(".//item"):
    link_tag = item.find("link")
    if link_tag is None or not link_tag.text:
        continue
    story_dict = {"url": link_tag.text.strip()}
    teasers_list.append(story_dict)

protothema_teasers_df = pd.DataFrame(teasers_list)
protothema_teasers_df = protothema_teasers_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
protothema_teasers_df


,url
0,https://www.protothema.gr/technology/article/1...
1,https://www.protothema.gr/technology/article/1...
2,https://www.protothema.gr/technology/article/1...
3,https://www.protothema.gr/technology/article/1...
4,https://www.protothema.gr/technology/article/1...
...,...
95,https://www.protothema.gr/technology/article/1...
96,https://www.protothema.gr/technology/article/1...
97,https://www.protothema.gr/technology/article/1...
98,https://www.protothema.gr/technology/article/1...


In [ ]:
# Αποθηκεύουμε τα teaser urls σε csv, ίδια λογική με το efsyn tutorial (μπορείς να αλλάξεις
# το path αν θες να το σώσεις στο Google Drive σου).
protothema_teasers_df.to_csv("protothema_teasers_technology_2026.csv", index=False)


## ΒΗΜΑ Β: Δοκιμή σε ένα άρθρο πριν φτιάξουμε τη λούπα

Ίδια λογική με το ΒΗΜΑ 5 του tutorial: πρώτα δοκιμάζουμε σε ένα άρθρο, βλέπουμε ότι δουλεύει, και
μετά φτιάχνουμε τη λούπα για όλα.


In [ ]:
article_url = protothema_teasers_df.loc[0, "url"]
print(article_url)

response = requests.get(article_url, headers=HEADERS)
doc = BeautifulSoup(response.text, "html.parser")

# Το html tag <article> συνήθως χρησιμοποιείται μόνο για το κυρίως άρθρο, οπότε -όπως λέει και
# το tutorial- δεν χρειάζεται να καταγράψουμε class. Αν σε κάποιο άρθρο δεν υπάρχει <article>,
# πέφτουμε πίσω σε ολόκληρο το doc.
article = doc.find("article") or doc
print(article.prettify()[:3000])


https://www.protothema.gr/technology/article/1880080/h-openai-apokalupse-nea-peristatika-me-tin-tehniti-noimosuni-oi-aprosdokites-i-anisuhitikes-suberifores-ton-modelon/?utm_source=rss
<article>
 <a class="mainLink" href="https://www.protothema.gr/world/article/1879227/proin-ereunitis-tis-google-bike-ston-horo-kai-proeidopoiise-gia-tin-apeili-tis-tehnitis-noimosunis/">
 </a>
 <figure data-image-mode="article">
  <a href="https://www.protothema.gr/world/article/1879227/proin-ereunitis-tis-google-bike-ston-horo-kai-proeidopoiise-gia-tin-apeili-tis-tehnitis-noimosunis/">
   <picture>
    <!--[if IE 9]><video style="display: none;"><![endif]-->
    <source class="lazysrcset" data-srcset="https://i1.prth.gr/images/304x304/2/jpg/files/2026-06-13/artificial_inteligence_xr.webp" media="(max-width: 639px)" type="image/webp"/>
    <source class="lazysrcset" data-srcset="https://i1.prth.gr/images/304x304/2/jpg/files/2026-06-13/artificial_inteligence_xr.jp2" media="(max-width: 639px)" type="image/

In [ ]:
# Τίτλος -- ο τίτλος συνήθως είναι σε h1 tag
title = article.find("h1").text.strip() if article.find("h1") else None
title


In [ ]:
# Ημερομηνία -- το protothema δεν έχει <time datetime=...> σαν το efsyn, έχει αυτό το meta
# tag στο <head> της σελίδας, με την ίδια λειτουργία (έτοιμη μορφή για pd.to_datetime).
meta_date = doc.find("meta", {"property": "article:published_time"})
date = meta_date["content"] if meta_date else None
date


'2026-09-17T09:50:00+03:00'

In [ ]:
# Συντάκτης δεν βρέθηκε ορατό byline tag/class στα δείγματα που ελέγχθηκαν, οπότε ψάχνουμε
# στα δομημένα δεδομένα JSON-LD του άρθρου (πολύ συχνό σε σύγχρονα news sites). Αν δεν υπάρχει,
# ο author μένει None -- δεν μαντεύουμε.
author = None
for script in doc.find_all("script", type="application/ld+json"):
    try:
        data = json.loads(script.string)
    except (TypeError, ValueError):
        continue
    if isinstance(data, dict) and "author" in data:
        a = data["author"]
        if isinstance(a, dict):
            author = a.get("name")
        elif isinstance(a, list) and a:
            author = a[0].get("name")
        break
author


In [ ]:
# μαζεύουμε όλα τα <p> tags
# μέχρι να συναντήσουμε μια από τις STOP_PHRASES.
p_texts_list = []
for p in article.find_all("p"):
    text = p.text.strip()
    if not text:
        continue
    if any(stop in text for stop in STOP_PHRASES):
        break
    p_texts_list.append(text)

full_text = " ".join(p_texts_list)
full_text = "".join(full_text.splitlines())
full_text


''

## ΒΗΜΑ Γ: Η λούπα για όλα τα άρθρα

Ίδια δομή με το tutorial: για κάθε url φτιάχνουμε ένα `story_dict`, το προσθέτουμε σε μια λίστα,
και στο τέλος τη μετατρέπουμε σε DataFrame.


In [ ]:
articles_list = []

for i, row in protothema_teasers_df.iterrows():
    url = row["url"]
    print(f"{i+1}/{len(protothema_teasers_df)}: {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        doc = BeautifulSoup(response.text, "html.parser")
        article = doc.find("article") or doc

        title = article.find("h1").text.strip() if article.find("h1") else None

        meta_date = doc.find("meta", {"property": "article:published_time"})
        date = meta_date["content"] if meta_date else None

        author = None
        for script in doc.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
            except (TypeError, ValueError):
                continue
            if isinstance(data, dict) and "author" in data:
                a = data["author"]
                if isinstance(a, dict):
                    author = a.get("name")
                elif isinstance(a, list) and a:
                    author = a[0].get("name")
                break

        p_texts_list = []
        for p in article.find_all("p"):
            text = p.text.strip()
            if not text:
                continue
            if any(stop in text for stop in STOP_PHRASES):
                break
            p_texts_list.append(text)
        full_text = " ".join(p_texts_list)
        full_text = "".join(full_text.splitlines())

        story_dict = {
            "site": "protothema.gr",
            "url": url,
            "title": title,
            "date": date,
            "author": author,
            "full_text": full_text if full_text else None,
        }
    except Exception as e:
        print("  -> Σφάλμα:", e)
        story_dict = {
            "site": "protothema.gr", "url": url, "title": None,
            "date": None, "author": None, "full_text": None,
        }

    articles_list.append(story_dict)
    time.sleep(1)


1/100: https://www.protothema.gr/technology/article/1880080/h-openai-apokalupse-nea-peristatika-me-tin-tehniti-noimosuni-oi-aprosdokites-i-anisuhitikes-suberifores-ton-modelon/?utm_source=rss
2/100: https://www.protothema.gr/technology/article/1879087/tehniti-noimosuni-horis-frena-oi-dimiourgoi-tis-ai-arhisan-na-fovoudai/?utm_source=rss
3/100: https://www.protothema.gr/technology/article/1878018/i-ai-diavazei-tin-kardia-prin-nosisei-ti-apokaluptoun-mastografia-kai-fotografies-prosopou/?utm_source=rss
4/100: https://www.protothema.gr/technology/article/1876627/samsung-kata-apple-mas-adigrapsate-to-trolarisma-gia-to-neo-iphone/?utm_source=rss
5/100: https://www.protothema.gr/technology/article/1876521/to-iphone-duo-pou-diplonei-kai-kostizei-2399-euro-epanastatis-i-ouragos/?utm_source=rss
6/100: https://www.protothema.gr/technology/article/1876359/apo-1999-to-neo-anadiploumeno-iphone-duo-pou-kukloforei-stis-16-oktovriou-sta-1199-to-iphone-18/?utm_source=rss
7/100: https://www.protothema.g

## ΒΗΜΑ Δ: DataFrame, φιλτράρισμα στο 2026, καθάρισμα

In [ ]:
protothema_df = pd.DataFrame(articles_list)

# Μετατροπή date (string) σε πραγματικό datetime -- ίδια λογική με το efsyn NLP notebook
# (pd.to_datetime), απλά εδώ το date string έχει ήδη ζώνη ώρας μέσα του (π.χ. "+03:00").
protothema_df["datetime"] = pd.to_datetime(protothema_df["date"], errors="coerce", utc=True)
protothema_df["datetime"] = protothema_df["datetime"].dt.tz_convert("Europe/Athens").dt.tz_localize(None)

protothema_df = protothema_df[["site", "url", "title", "date", "author", "full_text", "datetime"]]
protothema_df.head()


,site,url,title,date,author,full_text,datetime
0,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-17T09:50:00+03:00,None,None,2026-09-17 09:50:00
1,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-15T16:49:00+03:00,Δημήτρης Παγαδάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-15 16:49:00
2,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-13T10:48:00+03:00,None,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-13 10:48:00
3,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-10T10:46:00+03:00,None,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-10 10:46:00
4,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-10T07:09:00+03:00,Βασίλης Τσακίρογλου,None,2026-09-10 07:09:00


In [ ]:
protothema_2026_df = protothema_df[
    (protothema_df["datetime"] >= f"{target_year}-01-01")
    & (protothema_df["datetime"] < f"{target_year + 1}-01-01")
].copy()

protothema_2026_df = protothema_2026_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
protothema_2026_df = protothema_2026_df.sort_values("datetime").reset_index(drop=True)

print("Πλήθος άρθρων:", len(protothema_2026_df))
protothema_2026_df.head()


Πλήθος άρθρων: 92


,site,url,title,date,author,full_text,datetime
0,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-03T22:40:00+02:00,Κώστας Μαρτάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-03 22:40:00
1,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-05T17:58:00+02:00,None,None,2026-01-05 17:58:00
2,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-06T16:51:00+02:00,None,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-06 16:51:00
3,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-09T15:14:08+02:00,None,None,2026-01-09 15:14:08
4,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-10T08:30:02+02:00,None,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-10 08:30:02


In [ ]:
start_page = 1
end_page = 5
start_url = "https://www.protothema.gr/technology/???page="   # <- βάλε το σωστό pattern εδώ

extra_teasers = []
for page_num in range(start_page, end_page + 1):
    page_url = start_url + str(page_num)
    response = requests.get(page_url, headers=HEADERS)
    doc = BeautifulSoup(response.text, "html.parser")
    for a in doc.find_all("a", href=True):
        href = a["href"]
        if "/technology/article/" in href:
            full_url = href if href.startswith("http") else main_url + href
            extra_teasers.append({"url": full_url})
    time.sleep(1)

extra_teasers_df = pd.DataFrame(extra_teasers).drop_duplicates(subset=["url"])
print(len(extra_teasers_df))

19


## Αποθήκευση

In [ ]:
protothema_2026_df.to_csv("protothema_technology_2026.csv", index=False, encoding="utf-8-sig")
print("Αποθηκεύτηκε: protothema_technology_2026.csv (%d γραμμές)" % len(protothema_2026_df))


Αποθηκεύτηκε: protothema_technology_2026.csv (92 γραμμές)


## Γρήγορος έλεγχος

In [ ]:
print("Σύνολο άρθρων:", len(protothema_2026_df))
print("\nΆρθρα ανά μήνα:")
print(protothema_2026_df["datetime"].dt.to_period("M").value_counts().sort_index())
print("\nMin datetime:", protothema_2026_df["datetime"].min())
print("Max datetime:", protothema_2026_df["datetime"].max())
print("\nMissing values ανά στήλη:")
print(protothema_2026_df.isna().sum())

n = min(10, len(protothema_2026_df))
print(f"\n{n} τυχαία urls:")
for u in protothema_2026_df["url"].sample(n=n).tolist():
    print(" -", u)

n2 = min(5, len(protothema_2026_df))
print(f"\n{n2} τυχαία δείγματα:")
for _, row in protothema_2026_df.sample(n=n2).iterrows():
    print("\n---")
    print("Title:", row["title"])
    print("Author:", row["author"])
    print("Datetime:", row["datetime"])
    print("Text[:500]:", (row["full_text"] or "")[:500])


Σύνολο άρθρων: 92

Άρθρα ανά μήνα:
datetime
2026-01    24
2026-02    14
2026-03     7
2026-04     9
2026-05    12
2026-06     5
2026-07    10
2026-08     2
2026-09     9
Freq: M, Name: count, dtype: int64

Min datetime: 2026-01-03 22:40:00
Max datetime: 2026-09-17 09:50:00

Missing values ανά στήλη:
site          0
url           0
title        92
date          0
author       84
full_text    29
datetime      0
dtype: int64

10 τυχαία urls:
 - https://www.protothema.gr/technology/article/1786381/giati-oi-ai-agents-den-apoteloun-mono-ena-chatbot/?utm_source=rss
 - https://www.protothema.gr/technology/article/1790401/ti-deihnoun-oi-sunomilies-metaxu-ai-agent-kai-hristi/?utm_source=rss
 - https://www.protothema.gr/technology/article/1786379/pos-oi-ai-agents-apoktoun-leitourgiko-rolo/?utm_source=rss
 - https://www.protothema.gr/technology/article/1848762/i-ee-katigorei-ti-meta-gia-ethistiko-shediasmo-se-instagram-kai-facebook-kai-kinduneuei-me-prostimo/?utm_source=rss
 - https://www.protothe

In [ ]:
extra_articles_list = []

for i, row in extra_teasers_df.iterrows():
    url = row["url"]
    print(f"{i+1}/{len(extra_teasers_df)}: {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        doc = BeautifulSoup(response.text, "html.parser")

        # Use 'article' tag or the whole doc if not found, as in the protothema scraping logic
        article = doc.find("article") or doc

        full_article_dict = {}

        # website
        full_article_dict["site"] = "protothema.gr"

        # url
        full_article_dict["url"] = url

        # title
        title = article.find("h1").text.strip() if article.find("h1") else None
        full_article_dict["title"] = title

        # date from meta tag
        meta_date = doc.find("meta", {"property": "article:published_time"})
        date = meta_date["content"] if meta_date else None
        full_article_dict["date"] = date

        # author from JSON-LD
        author = None
        for script in doc.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
            except (TypeError, ValueError):
                continue
            if isinstance(data, dict) and "author" in data:
                a = data["author"]
                if isinstance(a, dict):
                    author = a.get("name")
                elif isinstance(a, list) and a:
                    author = a[0].get("name")
                break
        full_article_dict["author"] = author

        # full_text
        p_texts_list = []
        for p in article.find_all("p"):
            text = p.text.strip()
            if not text:
                continue
            if any(stop in text for stop in STOP_PHRASES):
                break
            p_texts_list.append(text)
        full_text = " ".join(p_texts_list)
        full_text = "".join(full_text.splitlines())
        full_article_dict["full_text"] = full_text if full_text else None

    except Exception as e:
        print("  -> Σφάλμα:", e)
        full_article_dict = {
            "site": "protothema.gr", "url": url, "title": None,
            "date": None, "author": None, "full_text": None,
        }

    extra_articles_list.append(full_article_dict)
    time.sleep(1)

protothema_extra_articles_df = pd.DataFrame(extra_articles_list)

# Convert date (string) to actual datetime
protothema_extra_articles_df["datetime"] = pd.to_datetime(protothema_extra_articles_df["date"], errors="coerce", utc=True)
protothema_extra_articles_df["datetime"] = protothema_extra_articles_df["datetime"].dt.tz_convert("Europe/Athens").dt.tz_localize(None)

protothema_extra_articles_df = protothema_extra_articles_df[["site", "url", "title", "date", "author", "full_text", "datetime"]]
protothema_extra_articles_df.head()

1/19: https://www.protothema.gr/technology/article/1880810/ereunites-kuvernoasfaleias-hakaran-tin-openai-me-ergaleio-tis-anthropic-nees-anisuhies-gia-tin-prostasia-ton-modelon-ai/
5/19: https://www.protothema.gr/technology/article/1880080/h-openai-apokalupse-nea-peristatika-me-tin-tehniti-noimosuni-oi-aprosdokites-i-anisuhitikes-suberifores-ton-modelon/
10/19: https://www.protothema.gr/technology/article/1879087/tehniti-noimosuni-horis-frena-oi-dimiourgoi-tis-ai-arhisan-na-fovoudai/
15/19: https://www.protothema.gr/technology/article/1878986/o-bil-geits-proeidopoiei-kamia-kuvernisi-ston-kosmo-den-einai-etoimi-gia-tis-allages-pou-tha-ferei-i-tehniti-noimosuni/
20/19: https://www.protothema.gr/technology/article/1878881/trab-kata-ton-periorismon-stin-ai-ta-robot-den-tha-katalavoun-ton-kosmo/
24/19: https://www.protothema.gr/technology/article/1878018/i-ai-diavazei-tin-kardia-prin-nosisei-ti-apokaluptoun-mastografia-kai-fotografies-prosopou/
27/19: https://www.protothema.gr/technology/art

,site,url,title,date,author,full_text,datetime
0,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-18T14:02:00+03:00,None,None,2026-09-18 14:02:00
1,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-17T09:50:00+03:00,None,None,2026-09-17 09:50:00
2,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-15T16:49:00+03:00,Δημήτρης Παγαδάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-15 16:49:00
3,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-15T10:33:00+03:00,None,None,2026-09-15 10:33:00
4,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-09-15T07:05:00+03:00,None,None,2026-09-15 07:05:00


In [ ]:
protothema_2026_df.loc[0, 'full_text']

'Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συνδυάζει διαφορετικές ασφαλιστικές ανάγκες και προσφέρει κλιμακωτές εκπτώσεις στα ασφάλιστρα'

In [ ]:
protothema_2026_df.loc[0, 'date']

'2026-01-03T22:40:00+02:00'

In [ ]:
protothema_2026_df['clean_text'] = protothema_2026_df['full_text'].str.replace(
    "Ακολουθήστε το πρωτοθέμα στο Google News", "", regex=False
)

In [ ]:
protothema_2026_df['datetime'] = pd.to_datetime(
    protothema_2026_df['date'], format='%Y-%m-%dT%H:%M:%S%z'
)
protothema_2026_df.head()

/tmp/ipykernel_780/1484539430.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  protothema_2026_df['datetime'] = pd.to_datetime(


,site,url,title,date,author,full_text,datetime,clean_text
0,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-03T22:40:00+02:00,Κώστας Μαρτάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-03 22:40:00+02:00,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...
1,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-05T17:58:00+02:00,None,None,2026-01-05 17:58:00+02:00,None
2,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-06T16:51:00+02:00,None,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-06 16:51:00+02:00,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...
3,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-09T15:14:08+02:00,None,None,2026-01-09 15:14:08+02:00,None
4,protothema.gr,https://www.protothema.gr/technology/article/1...,None,2026-01-10T08:30:02+02:00,None,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-10 08:30:02+02:00,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...


In [ ]:
protothema_2026_df = pd.read_csv('protothema_technology_2026.csv')
protothema_2026_df['datetime'] = pd.to_datetime(
    protothema_2026_df['date'], format='%Y-%m-%dT%H:%M:%S%z'
)

/tmp/ipykernel_780/2108615219.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  protothema_2026_df['datetime'] = pd.to_datetime(


In [ ]:
print(protothema_2026_df.shape)
protothema_2026_df

(92, 7)


,site,url,title,date,author,full_text,datetime
0,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-03T22:40:00+02:00,Κώστας Μαρτάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-03 22:40:00+02:00
1,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-05T17:58:00+02:00,NaN,NaN,2026-01-05 17:58:00+02:00
2,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-06T16:51:00+02:00,NaN,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-06 16:51:00+02:00
3,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-09T15:14:08+02:00,NaN,NaN,2026-01-09 15:14:08+02:00
4,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-10T08:30:02+02:00,NaN,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-10 08:30:02+02:00
...,...,...,...,...,...,...,...
87,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-09-10T07:09:00+03:00,Βασίλης Τσακίρογλου,NaN,2026-09-10 07:09:00+03:00
88,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-09-10T10:46:00+03:00,NaN,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-10 10:46:00+03:00
89,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-09-13T10:48:00+03:00,NaN,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-13 10:48:00+03:00
90,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-09-15T16:49:00+03:00,Δημήτρης Παγαδάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-09-15 16:49:00+03:00


In [ ]:
protothema_2026_df.to_csv("protothema_technology_2026.csv", index=False, encoding="utf-8-sig")

In [ ]:
import base64
import requests
from google.colab import userdata

def save_df_to_github(df, repo, path, token=None, branch="main", message="Update dataset"):
    token = token or userdata.get("GITHUB_TOKEN")
    csv_content = df.to_csv(index=False, encoding="utf-8-sig")
    content_b64 = base64.b64encode(csv_content.encode("utf-8-sig")).decode("utf-8")

    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

    existing = requests.get(url, headers=headers, params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None

    payload = {"message": message, "content": content_b64, "branch": branch}
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=headers, json=payload)
    response.raise_for_status()
    print(f"✅ Αποθηκεύτηκε: https://github.com/{repo}/blob/{branch}/{path}")
    return response.json()

save_df_to_github(
    protothema_2026_df,
    repo="annatsamoyra-prog/data-story-",
    path="protothema_technology_2026.csv",
    token=userdata.get("newtoken")
)

✅ Αποθηκεύτηκε: https://github.com/annatsamoyra-prog/data-story-/blob/main/protothema_technology_2026.csv


{'content': {'name': 'protothema_technology_2026.csv',
  'path': 'protothema_technology_2026.csv',
  'sha': 'b2f238ad9826c986955760c9a6a87bb5c728574a',
  'size': 35431,
  'url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/protothema_technology_2026.csv?ref=main',
  'html_url': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/protothema_technology_2026.csv',
  'git_url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/b2f238ad9826c986955760c9a6a87bb5c728574a',
  'download_url': 'https://raw.githubusercontent.com/annatsamoyra-prog/data-story-/main/protothema_technology_2026.csv',
  'type': 'file',
  '_links': {'self': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/protothema_technology_2026.csv?ref=main',
   'git': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/b2f238ad9826c986955760c9a6a87bb5c728574a',
   'html': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/protothema_te

In [38]:
nan_rows = protothema_2026_df[protothema_2026_df['full_text'].isna()]
nan_rows

,site,url,title,date,author,full_text,datetime


In [36]:
nan_rows = protothema_2026_df[protothema_2026_df['full_text'].isna()]
protothema_2026_df = protothema_2026_df.drop(nan_rows.index)
protothema_2026_df.to_csv("protothema_technology_2026.csv", index=False, encoding="utf-8-sig")

In [39]:
duplicates = protothema_2026_df.duplicated(subset=['full_text']).any()
duplicates

np.True_

In [42]:
protothema_2026_df = protothema_2026_df.drop_duplicates(subset='full_text', keep='first').reset_index(drop=True)

In [44]:
# βεβαιώνομαι οτι η ημερομηνία μου έχει format datetime
protothema_2026_df['date'] = pd.to_datetime(protothema_2026_df['date'], errors='coerce')

In [46]:
protothema_2026_df['text'] = protothema_2026_df['full_text'].fillna('')

In [47]:

#εισάγω τη βιβιοθήκη re για δημιουργία regex (regular expressions)-δηλαδή κανόνες αναζήτησης και εντοπισμού μοτίβων
import re

In [48]:
# δημιουργώ regex patterns που εντοπίζουν προβληματικά μοτίβα για βασικό καθαρισμό κειμένου (δηλ. τι ψάχνω)

url      = re.compile(r'https?://\S+|www\.\S+')  # εντοπίζει URLs/links
mentions  = re.compile(r'@\w+')                  # εντοπίζει mentions π.χ. @username
dashes   = re.compile(r'[-–—−]+')                # εντοπίζει όλα τα είδη παύλας (– — − κτλ.) και τα κανονικοποιεί
quotes   = re.compile(r'[“”]')                   # εντοπίζει “curly” διπλά εισαγωγικά
apos     = re.compile(r'[’]')                    # εντοπιζει το curly απόστροφο (’)
spaces   = re.compile(r'\s+')                    # εντοπίζει πολλά συνεχόμενα κενά

In [50]:
# φτιάχνω συνάρτηση που εφαρμόζει τα παραπάνω μοτίβα και κάνει την πραγματική κανονικοποίηση του κειμένου (δηλ. τι κάνω όταν εντοπίσω το πρόβλημα)
# ηπιος καθαρισμός, δεν αφαιρούμε τη στίξη

def mild_text_clean(text: str) -> str:

    text = str(text)

    text = url.sub(" ", text)        # αφαιρεί links
    text = mentions.sub(" ", text)    # αφαιρεί mentions
    text = dashes.sub(" — ", text)   # ενοποιεί τις παύλες σε em dash
    text = quotes.sub('"', text)     # κάνει “ ” → "
    text = apos.sub("'", text)       # κάνει ’ → '
    text = spaces.sub(" ", text).strip()  # αφαιρεί περιττά κενά

    return text


In [51]:
#εφαρμοζω τη συναρτηση και αποθηκεύω σε νέα στήλη

protothema_2026_df["text_clean"] = protothema_2026_df["text"].apply(mild_text_clean)

In [52]:
# check
protothema_2026_df.head()


,site,url,title,date,author,full_text,datetime,text,text_clean
0,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-03 22:40:00+02:00,Κώστας Μαρτάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-03 22:40:00+02:00,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...


In [55]:
import string

def clean_text(article: str) -> str:
    s = str(article).lower() # mετατρέπει το κείμενο σε string και όλα τα γράμματα σε πεζά
    s = re.sub(r'[' + re.escape(string.punctuation) + r'’“”–—]', ' ', s) # αντικαθιστά όλα τα σημεία στίξης με κενό
    s = re.sub(r'\s+', ' ', s).strip() # αφαιρεί τα περιττά κενά ανάμεσα στις λέξεις

    return s

In [56]:
protothema_2026_df['tokenized'] = protothema_2026_df['text_clean'].map(lambda x: clean_text(x))

In [57]:
protothema_2026_df.tail

<bound method NDFrame.tail of             site                                                url  title  \
0  protothema.gr  https://www.protothema.gr/technology/article/1...    NaN   

                       date           author  \
0 2026-01-03 22:40:00+02:00  Κώστας Μαρτάκης   

                                           full_text  \
0  Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...   

                    datetime  \
0  2026-01-03 22:40:00+02:00   

                                                text  \
0  Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...   

                                          text_clean  \
0  Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...   

                                           tokenized  
0  το πρόγραμμα eurolifesyn της eurolife ffh συνδ...  >

In [59]:
import string

import nltk

# Αφαίρεση Stopwords

# εισάγω βιβλιοθήκες και εργαλεία για tokenization και stopwords

# η βιβλιοθήκη string περιέχει έτοιμους χαρακτήρες και εργαλεία για επεξεργασία κειμένου


#η nltk (Natural Language Toolkit) χρησιμοποιείται για βασικές εργασίες NLP (Natural Language Processing) όπως tokenization, stopwords κτλ.


# κατεβάζω το tokenizer μοντέλο punkt που χρησιμοποιείται για sentence splitting και tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

# παίρνω όλα τα σημεία στίξης και τα αποθηκεύω σε λίστα
punct = list(set(string.punctuation))

## εισάγω τα stopwords της NLTK
from nltk.corpus import stopwords

#εισάγω tokenizer για να χωρίσω το κείμενο σε λέξεις/tokens
from nltk.tokenize import word_tokenize

# κατεβάζω τη λίστα stopwords
nltk.download('stopwords')

## δημιουργώ set με τα αγγλικά stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [60]:

def remove_stopwords(x):

    # ελέγχω αν το input είναι ήδη λίστα από tokens ή κανονικό κείμενο/string
    tokens = x if isinstance(x, (list, tuple)) else word_tokenize(str(x))

    # κρατάω μόνο τις λέξεις που ΔΕΝ είναι stopwords
    return [t for t in tokens if t.lower() not in stop_words]

In [61]:
#εφαρμοζω τη συναρτηση και αποθηκεύω σε νέα στήλη

protothema_2026_df['no_stop'] = protothema_2026_df['tokenized'].apply(remove_stopwords)



In [62]:
protothema_2026_df.head(2)

,site,url,title,date,author,full_text,datetime,text,text_clean,tokenized,no_stop
0,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-03 22:40:00+02:00,Κώστας Μαρτάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-03 22:40:00+02:00,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,το πρόγραμμα eurolifesyn της eurolife ffh συνδ...,"[το, πρόγραμμα, eurolifesyn, της, eurolife, ff..."


In [63]:
protothema_2026_df['num_wds'] = protothema_2026_df['tokenized'].apply(lambda x: len(x.split()))#αποθηκεύω τον αριθμό λέξεων του άρθρου σε δική του στήλη
protothema_2026_df['num_wds'].mean()

np.float64(16.0)

In [64]:
# συχνότητα μοναδικών λέξεων ανα κείμενο
protothema_2026_df['uniq_wds'] = protothema_2026_df['tokenized'].str.split().apply(lambda x: len(set(x))) #αποθηκεύω τον αριθμό μοναδικών λεξεων του άρθρου σε δική του στήλη
protothema_2026_df['uniq_wds'].mean()


np.float64(16.0)

In [65]:
# δημιουργώ στήλη Text_id για να μην χάνω τα κείμενα
protothema_2026_df = protothema_2026_df.reset_index().rename(columns={'index': 'Text_id'})


In [66]:
protothema_2026_df.head()

,Text_id,site,url,title,date,author,full_text,datetime,text,text_clean,tokenized,no_stop,num_wds,uniq_wds
0,0,protothema.gr,https://www.protothema.gr/technology/article/1...,NaN,2026-01-03 22:40:00+02:00,Κώστας Μαρτάκης,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,2026-01-03 22:40:00+02:00,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,Το πρόγραμμα EurolifeSYN+ της Eurolife FFH συν...,το πρόγραμμα eurolifesyn της eurolife ffh συνδ...,"[το, πρόγραμμα, eurolifesyn, της, eurolife, ff...",16,16


In [67]:
protothema_2026_df.to_csv('https://github.com/annatsamoyra-prog/data-story-.git')

In [70]:
save_df_to_github(
    protothema_2026_df,
    repo="annatsamoyra-prog/data-story-",
    path="protothema_technology_2026_csv_clean.csv",
    token=userdata.get("newtoken")
)

✅ Αποθηκεύτηκε: https://github.com/annatsamoyra-prog/data-story-/blob/main/protothema_technology_2026_csv_clean.csv


{'content': {'name': 'protothema_technology_2026_csv_clean.csv',
  'path': 'protothema_technology_2026_csv_clean.csv',
  'sha': '66bb4833d30d899a7f7ff574e54f52d3e1fda13a',
  'size': 1591,
  'url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/protothema_technology_2026_csv_clean.csv?ref=main',
  'html_url': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/protothema_technology_2026_csv_clean.csv',
  'git_url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/66bb4833d30d899a7f7ff574e54f52d3e1fda13a',
  'download_url': 'https://raw.githubusercontent.com/annatsamoyra-prog/data-story-/main/protothema_technology_2026_csv_clean.csv',
  'type': 'file',
  '_links': {'self': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/protothema_technology_2026_csv_clean.csv?ref=main',
   'git': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/66bb4833d30d899a7f7ff574e54f52d3e1fda13a',
   'html': 'https://githu